# Data types

This page considers the data types in clickhouse.

## DateTime

The dates in clickhouse there are types that allows to represent time in clickhouse:

| Type                      | Storage | Range                         | Precision                   |
| ------------------------- | ------- | ----------------------------- | --------------------------- |
| `Date`                    | 2 bytes | 1970-01-01 – 2149-06-06       | Day                         |
| `Date32`                  | 4 bytes | 1900-01-01 – 2299-12-31       | Day                         |
| `DateTime`                | 4 bytes | 1970-01-01 – 2106-02-07       | Second                      |
| `DateTime64(p)`           | 8 bytes | 1900-01-01 – 2299-12-31       | Up to nanoseconds (`p=0–9`) |
| `Time`                    | 4 bytes | 00:00:00 – 23:59:59           | Second                      |
| `Time64(p)`               | 8 bytes | 00:00:00 – 23:59:59.999999999 | Up to nanoseconds (`p=0–9`) |
| [Interval type family][1] | 8 bytes | ±9,223,372,036,854,775,807    |                             |

[1]: https://clickhouse.com/docs/sql-reference/data-types/special-data-types/interval

The most usefull functions/operators to work with dates:

- The `toDate` function converts a string literar in the format `"yyyy-mm-dd"` into a `Date` object.
- The `dateDiff` function allows to count the period between two dates in the selected units. The minus operator `<date1> - <date2>` is not documented, but it appears to works in the same way as the `dateDiff('days', <date2>, <date1>)` function.
- The `INTERVAL <number> <SECOND | MINUTE | HOUR ...>` allows to create a special interval object that could be added to the date.
- The `now()` function return `DateTime` object that corresponds to current datetime.

May find usefull:
- [Operators for Working with Dates and Times](https://clickhouse.com/docs/sql-reference/operators#operators-for-working-with-dates-and-times).

---

Here are the example of usage of the most usable facilities associated with dates in clickhouse:

In [19]:
--ClickHouse
SELECT
    toDate('2025-08-01') AS date1,
    toDate('2025-07-01') AS date2,
    date1 - date2 AS diff,
    dateDiff('month', date2, date1) AS months,
    date1 + INTERVAL 10 DAY add_interval,
    now();

date1,date2,diff,months,add_interval,now()
2025-08-01,2025-07-01,31,31,2025-08-11,2026-06-08 19:06:43


## Array(T)

Array in clickhouse is a set of objects of the same type.

- Define array using `Array(<type>)`.
- Create inline array using `array(<el1>, <el2>, ...)` or using `[el1, el2, ...]`.
- Use the `sizeN` subcolumn to get the length of the array. The `N` determines the dimention of the array you're interested in. 

Check the description in the [Array(T)](https://clickhouse.com/docs/sql-reference/data-types/array) page of the official documentation.

---

The following cell defines the CTE containing the inline arrays and shows the size of their different diemntions.

In [1]:
--ClickHouse
WITH t_arr (arrays) AS(
    SELECT * FROM VALUES (
        [[1, 2, 3], [2, 3]]
    )
)
SELECT
    arrays.size0 AS first_diemntion,
    arrays.size1 AS second_dimention
FROM t_arr;

first_diemntion,second_dimention
2,"[3, 2]"


### Arithmetical operators

The use of the arithmetic operators with arrays as arguments is not documented. But in fact the `+` and `-` operartors work as a regular element-wise applition of the corresponding operators. Other operators, such as `*` or `/` cause the clickhouse error.

---

The following cell show the result of the `+` and `-` operators with numerical arrays as arguments:

In [29]:
--ClickHouse
SELECT
    [1, 2, 3] - [3, 2, 1],
    [1, 2, 3] + [3, 2, 1];

"minus([1, 2, 3], [3, 2, 1])","plus([1, 2, 3], [3, 2, 1])"
"[-2, 0, 2]","[4, 4, 4]"


The example for nested arrays:

In [31]:
--ClickHouse
SELECT
    [
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]
    ] AS matrix1,
    [
        [6, 2, 1],
        [7, 9, 3],
        [0, 2, 8]
    ] AS matrix2,
    matrix1 + matrix2,
    matrix2 - matrix1;

matrix1,matrix2,"plus(matrix1, matrix2)","minus(matrix2, matrix1)"
"[[1, 2, 3], [4, 5, 6], [7, 8, 9]]","[[6, 2, 1], [7, 9, 3], [0, 2, 8]]","[[7, 4, 4], [11, 14, 9], [7, 10, 17]]","[[5, 0, -2], [3, 4, -3], [-7, -6, -1]]"


## Tuple(T1, T2, ...)

A tuple is an ordered set of elements, each of which has a special data type.

- Define a column to have a `Tuple` data type using syntax `Typle(<type1>, <type2>, ...)`. You can assign a names to the elements of the typle using syntax `Typle(<name1> <type1>, <name2> <type2>, ...)`.
- Create inline tuple with `typle(<val1>, <val2>, ...)` or `(<val1>, <val2>, ...)`.
- Access the elements of a tuple using `.` operator, specifying the name or index of the element. For example, use `typ.1` to access the first element of the tuple, or `typ.a` to access the element named `a`.

Check the official description in the [Tuple](https://clickhouse.com/docs/sql-reference/data-types/tuple) page of the official documentation.

---

The following cell defines the typle in the CTE and accesses its elements.

In [21]:
--ClickHouse
WITH tuple_example AS (
    SELECT ('hello', 10) AS the_tuple
)
SELECT the_tuple, the_tuple.1, the_tuple.2 FROM tuple_example;

the_tuple,"tupleElement(the_tuple, 1)","tupleElement(the_tuple, 2)"
"('hello', 10)",hello,10


## Map(K, V)

Clickhouse implements mappings as an array of tuples: `Array(Tuple(T_key, T_val))`, which means their keys are not unique.

- Define column type with `Map(<key_type>, <value type>)`.
- Access elements using `[]` operator.
- Define the map inline using `map(<key1>, <value1>, <key2>, <value2>, ...)` or `{<key1>: <value1>, <key2>: <value2> ...}`.
- The `keys` and `values` columns return the keys and values `Array`.

---

The following cell defines a ClickHouse map and demonstrates some fundamental operations in queries.

In [8]:
--ClickHouse
WITH map_example AS (
    SELECT map('one', 1, 'two', 2) AS my_map
)
SELECT my_map['one'], my_map.keys, my_map.values FROM map_example;

"arrayElement(my_map, 'one')",my_map.keys,my_map.values
1,"['one', 'two']","[1, 2]"


## Dynamic

The [`Dynamic`](https://clickhouse.com/docs/sql-reference/data-types/dynamic) is a datatype to store the objects without knowing their specific datatype.

Typically `Dynamic` could be result of some operation with undefined output type. For example, extracting attributes from json produces `Dynamic` object.

---

The following cell defines the table with column `a` denoted as `Dynamic`.

In [5]:
--ClickHouse
CREATE OR REPLACE TABLE dynamic_example (a Dynamic);
INSERT INTO dynamic_example values (NULL), (10), ('string');

SELECT a, toTypeName(a) FROM dynamic_example;

a,toTypeName(a)
,Dynamic
10,Dynamic
string,Dynamic


Use the `dymanicType` function to find out what type is actually hiding behind each value:

In [20]:
--ClickHouse
SELECT dynamicType(a) FROM dynamic_example;

dynamicType(a)
None
Int64
String


So it can keep the objects with various datatypes.

### Casting

When working with `Dynamic`, it is crucial to know how the actual data could be extracted. There are several options:

- Casting: `cast` or `<column>::<type>` syntax will attempt to transform all the values to the specified datatype.
- Extracting nested types: `<dynamic_column>.<type>`. All values belonging to the specific will be included in the result, the others will have a `NULL` value.

---

The following cell casts the `Dynamic` column to the string:

In [10]:
--ClickHouse
SELECT
    cast(a, 'String') as casted_to_string,
    toTypeName(casted_to_string)
from dynamic_example;

casted_to_string,toTypeName(casted_to_string)
,String
10,String
string,String


All goes well - originally integer `10` is represented as a `String` type. But, as value `string` could not be converted to the integer, a corresponding error would be caused:

```sql
SELECT
    cast(a, 'Int64') as casted_to_string,
    toTypeName(casted_to_string)
from dynamic_example;
```

```
raise OperationalError(err_str) if retried else DatabaseError(err_str) from None
clickhouse_connect.driver.exceptions.DatabaseError: Received ClickHouse exception, code: 6, server response: Code: 6. DB::Exception: Cannot parse string 'string' as UInt8: syntax error at begin of string. Note: there are toUInt8OrZero and toUInt8OrNull functions, which returns zero/NULL instead of throwing exception: while executing 'FUNCTION CAST(__table1.a :: 3, 'UInt8'_String :: 1) -> CAST(__table1.a, 'UInt8'_String) UInt8 : 0'. (CANNOT_PARSE_TEXT) (for url http://localhost:55873)
```

Alternagive approach is to specify the type after the `.` operator. The following cell applies the `.` operator with different types:

In [24]:
--ClickHouse
SELECT
    a.Int64,
    a.String
FROM dynamic_example;

a.Int64,a.String
,
10,
,string


If the values have a corresponding type, the value is extracted. Otherwise, the `NULL` is the output.

**Note** you have to specify the exact type. Different accuracy representations of the same type would be interpreted as `NULL`:

In [25]:
--ClickHouse
SELECT
    a.Int8,
    a.UInt64,
    a.Int64
FROM dynamic_example;

a.Int8,a.UInt64,a.Int64
,,
,,10
,,


## JSON

ClickHouse natively supports the JSON data type. There are:

- Corresponding operators and functions for processing JSON data in ClickHouse.
- There are plenty options for creating a JSON dtype in the query.
- A lot of options to declare the JSON datatype column.

Check more in the [JSON Data Type](https://clickhouse.com/docs/sql-reference/data-types/newjson) manual page.

---

The definition of the table with JSON column and inserting some data the:

In [16]:
--ClickHouse
CREATE OR REPLACE TABLE json_example (col JSON); 
INSERT INTO json_example VALUES ('{"a": {"b": 10}}'), ('{"c": [1, 2, 3]}');
SELECT col, toTypeName(col) FROM json_example;

col,toTypeName(col)
{'a': {'b': 10}},JSON
"{'c': [1, 2, 3]}",JSON


You can refer to the columns really natively by simply specifying them as sub-columns:

In [14]:
--ClickHouse
SELECT col.a.b FROM json_example;

col.a.b
10
""
